# Aquarium Detection — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s6-aquarium-detection.ipynb)

Find every animal in an aquarium photo: a **bounding box** and a **class** for each, among
**7 classes** (fish, jellyfish, penguin, puffin, shark, starfish, stingray):
[challenge 195](https://ml-arena.com/viewchallenge/195) on ML-Arena. Metric: **mAP50-95**.

This notebook runs end-to-end: **download the data → look at the boxes → fine-tune a pretrained
YOLO → score it on held-out images → submit**.

**Part 2** then takes YOLO apart: decoding knobs, a bug hunt, broken training recipes,
the raw output tensor, and a YOLOv1 small enough to read line by line.

> In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
> The only thing you must edit is your API token (your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk>=3" ultralytics

import mlarena
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random, shutil, sys, zipfile, torch
from pathlib import Path
from PIL import Image

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
CHALLENGE_ID = 195

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`train.zip` holds 448 images and their labels, `test_images.zip` the 190 images you predict on.
`detection_metric.py` is the leaderboard's parser and metric: we import it to score locally.

In [ ]:
client.download_dataset(CHALLENGE_ID, "data/")
zipfile.ZipFile("data/train.zip").extractall("data/train")
zipfile.ZipFile("data/test_images.zip").extractall("data/test")
sys.path.insert(0, "data")
import detection_metric as dm

class_names = pd.read_csv("data/classes.csv")["name"].tolist()
train_ids = sorted(p.stem for p in Path("data/train/images").glob("*.jpg"))
test_images = sorted(Path("data/test/images").glob("*.jpg"))
print(len(train_ids), "training images,", len(test_images), "test images")
print(class_names)

## 2. Look at the boxes

A label file has one line per box: `class_id x_center y_center width height`, each coordinate
divided by the image's width or height (the **YOLO format**).

In [ ]:
def read_labels(image_id):
    lines = Path(f"data/train/labels/{image_id}.txt").read_text().split()
    return np.array(lines, dtype=float).reshape(-1, 5)

def draw(ax, image_path, boxes, title=""):
    img = Image.open(image_path)
    W, H = img.size
    ax.imshow(img)
    for cls, xc, yc, w, h in boxes[:, [0, -4, -3, -2, -1]]:
        color = plt.cm.tab10(int(cls))
        ax.add_patch(patches.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H,
                                       linewidth=2, edgecolor=color, facecolor="none"))
        ax.text((xc - w / 2) * W, (yc - h / 2) * H, class_names[int(cls)], color="white",
                fontsize=7, bbox=dict(facecolor=color, pad=1, linewidth=0))
    ax.set_title(title, fontsize=9); ax.axis("off")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, iid in zip(axes.flat, random.Random(0).sample(train_ids, 6)):
    boxes = read_labels(iid)
    draw(ax, f"data/train/images/{iid}.jpg", boxes, f"{len(boxes)} boxes")
plt.tight_layout(); plt.show()

counts = pd.Series(np.concatenate([read_labels(i)[:, 0] for i in train_ids]).astype(int))
print(counts.map(dict(enumerate(class_names))).value_counts())

**Questions**
- How many animals does an image hold? Which classes are rare, and why does that matter for a
  metric that averages over classes?
- Which boxes look hard: small, overlapping, cut by the image border?

## 3. Hold out a validation set

The test labels are private, so we keep 15 % of the training images aside to measure the model
before submitting, and write the `data.yaml` Ultralytics trains from.

In [ ]:
ids = train_ids[:]
random.Random(0).shuffle(ids)
split = {"val": ids[:len(ids) * 15 // 100], "train": ids[len(ids) * 15 // 100:]}
for name, part in split.items():
    for sub in ("images", "labels"):
        Path(f"yolo/{name}/{sub}").mkdir(parents=True, exist_ok=True)
    for iid in part:
        shutil.copy(f"data/train/images/{iid}.jpg", f"yolo/{name}/images/")
        shutil.copy(f"data/train/labels/{iid}.txt", f"yolo/{name}/labels/")
Path("aquarium.yaml").write_text(
    f"path: {Path('yolo').resolve()}\ntrain: train/images\nval: val/images\nnames: {class_names}\n")
print({k: len(v) for k, v in split.items()})

## 4. Fine-tune a pretrained YOLO

`yolov8n.pt` is the smallest YOLOv8, pretrained on COCO (80 everyday classes, no starfish).
Fine-tuning keeps its backbone and adapts the detection head to our 7 classes.

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO("yolov8n.pt")
model.train(data="aquarium.yaml", epochs=30, imgsz=640, batch=16, seed=0, device=device)
model = YOLO(Path(model.trainer.save_dir) / "weights" / "best.pt")   # best validation epoch

## 5. Score it with the leaderboard's metric

For mAP, predict with a **low confidence threshold** (0.001): low-confidence boxes cost nothing
at the top of the ranking and add recall at the bottom.

In [ ]:
def detect(model, paths, **predict_args):
    """{image_id: (n, 6) array}: class, confidence, then the box normalised as in the labels."""
    predict_args = {"conf": 0.001, **predict_args}
    out = {}
    for i in range(0, len(paths), 16):
        batch = paths[i:i + 16]
        for p, r in zip(batch, model.predict([str(p) for p in batch], device=device, verbose=False, **predict_args)):
            b = r.boxes
            out[Path(p).stem] = np.column_stack([b.cls.cpu().numpy(), b.conf.cpu().numpy(), b.xywhn.cpu().numpy()]).reshape(-1, 6)
    return out

val_images = sorted(Path("yolo/val/images").glob("*.jpg"))
val_gt = {iid: read_labels(iid) for iid in split["val"]}
val_pred = detect(model, val_images)
scores = dm.evaluate(val_gt, val_pred)
print("validation mAP50-95 %.4f   mAP50 %.4f" % (scores["map50_95"], scores["map50"]))
print({c: round(scores[f"ap_{c}"], 3) for c in class_names})

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, iid in enumerate(split["val"][:3]):
    draw(axes[0, col], f"yolo/val/images/{iid}.jpg", val_gt[iid], "ground truth")
    shown = val_pred[iid][val_pred[iid][:, 1] >= 0.25]   # draw the confident ones only
    draw(axes[1, col], f"yolo/val/images/{iid}.jpg", shown, "predicted, confidence >= 0.25")
plt.tight_layout(); plt.show()

**Questions**
- Which class has the lowest AP? Is it rare in the training set, small, or confused with another?
- What does the gap between mAP50 and mAP50-95 tell you about the boxes?

## 6. Predict the test images and submit

In [ ]:
test_pred = detect(model, test_images)
submission = pd.DataFrame({"image_id": list(test_pred),
                           "prediction_string": [dm.to_prediction_string(b) for b in test_pred.values()]})
submission.to_csv("submission.csv", index=False)
submission.head()

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"], submission_name="yolov8n-30ep")
print(result)

# Part 2 — Take YOLO apart

The benchmark is on the board, in four calls: `YOLO(...)`, `train`, `predict`, `submit`.
Each one hides a piece of the *YOLO* lesson. The rest of the notebook opens them one by one:
the decoding knobs (seconds), a bug hunt, broken training recipes (about 2 minutes each),
the raw output tensor, and finally a YOLOv1 small enough to read in full.

**One rule for every experiment: write your prediction down before you run the cell.**
Higher, lower or the same as the reference, and roughly by how much. Being surprised
is the point; you can only be surprised if you predicted something.

## 7. The decoding knobs (no training)

Same weights every time. Only the decoding of the lesson changes: the score threshold
`conf`, the NMS threshold `iou`, the cap `max_det`, and the input size `imgsz`.

In [ ]:
def score(m, **predict_args):
    s = dm.evaluate(val_gt, detect(m, val_images, **predict_args))
    return {"mAP50-95": round(s["map50_95"], 4), "mAP50": round(s["map50"], 4), "boxes": s["n_detections"]}

knobs = {
    "reference (conf=0.001, iou=0.7)": {},
    "conf=0.25":    {"conf": 0.25},
    "conf=0.5":     {"conf": 0.5},
    "iou=0.3":      {"iou": 0.3},
    "iou=0.99":     {"iou": 0.99},
    "agnostic_nms": {"agnostic_nms": True},
    "max_det=10":   {"max_det": 10},
    "imgsz=320":    {"imgsz": 320},
    "imgsz=960":    {"imgsz": 960},
}
pd.DataFrame({name: score(model, **kw) for name, kw in knobs.items()}).T

**Questions**
- `conf=0.25` draws cleaner pictures and scores lower. From the definition of AP, why can
  a low-confidence box never lower it?
- `iou` is the NMS threshold. What survives NMS at 0.99? Why does 0.3 also cost a little,
  on images full of overlapping fish?
- `agnostic_nms=True` suppresses across classes. Why does it change almost nothing here,
  and on what kind of dataset would it matter?
- One validation image holds 55 animals. What does `max_det=10` cut, recall or precision?
- The model was trained at 640. Why does predicting at 320 cost so much, and why does 960
  bring nothing? (The images are 640 px on their long side; the finest stride is 8.)

## 8. Find the bug

A teammate wrote their own export. It runs, the leaderboard accepts the file, and it
scores about zero. One word differs from `detect`. Find it, then explain with `draw` on a
validation image what the metric thinks the four numbers are.

In [ ]:
def detect_teammate(m, paths):
    out = {}
    for i in range(0, len(paths), 16):
        batch = paths[i:i + 16]
        for p, r in zip(batch, m.predict([str(p) for p in batch], conf=0.001, device=device, verbose=False)):
            b = r.boxes
            out[Path(p).stem] = np.column_stack([b.cls.cpu().numpy(), b.conf.cpu().numpy(), b.xyxyn.cpu().numpy()]).reshape(-1, 6)
    return out

print("teammate's mAP50-95: %.4f" % dm.evaluate(val_gt, detect_teammate(model, val_images))["map50_95"])

## 9. Break the training (about 2 minutes per run on a T4)

Each recipe changes one thing in section 4 and trains for **10 epochs**. `reference` is
section 4's recipe for the same 10 epochs: compare every run with it, not with the
30-epoch benchmark. Run them one at a time, prediction first.

`results.csv` logs the three terms of the YOLOv8 loss from the lesson:
`box_loss` (CIoU), `cls_loss` (BCE) and `dfl_loss` (DFL), on train and on validation.

In [ ]:
recipes = {
    "reference":       {},
    "3 epochs":        {"epochs": 3},
    "from scratch":    {"model": "yolov8n.yaml"},   # same architecture, random weights
    # optimizer="auto" (the default) picks its own learning rate and ignores lr0
    "lr too high":     {"optimizer": "SGD", "lr0": 1.0},
    "lr too low":      {"optimizer": "SGD", "lr0": 1e-5},
    "tiny images":     {"imgsz": 160},
    "no augmentation": {"mosaic": 0, "fliplr": 0, "hsv_h": 0, "hsv_s": 0, "hsv_v": 0, "scale": 0, "translate": 0},
    "head only":       {"freeze": 22},              # backbone and neck frozen
}
board, logs = {}, {}

def experiment(name):
    args = {"model": "yolov8n.pt", "epochs": 10, "imgsz": 640, **recipes[name]}
    m = YOLO(args.pop("model"))
    m.train(data="aquarium.yaml", batch=16, seed=0, device=device, name=name.replace(" ", "_"),
            exist_ok=True, verbose=False, **args)
    logs[name] = pd.read_csv(Path(m.trainer.save_dir) / "results.csv").rename(columns=str.strip)
    best = YOLO(Path(m.trainer.save_dir) / "weights" / "best.pt")
    last = logs[name].iloc[-1]
    board[name] = {**score(best, imgsz=args["imgsz"]),
                   "train box_loss": round(last["train/box_loss"], 3), "val box_loss": round(last["val/box_loss"], 3)}
    return pd.DataFrame(board).T

def plot_losses(*names):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3))
    for ax, term in zip(axes, ("box_loss", "cls_loss", "dfl_loss")):
        for name in names:
            ax.plot(logs[name]["epoch"], logs[name][f"train/{term}"], label=f"{name} train")
            ax.plot(logs[name]["epoch"], logs[name][f"val/{term}"], "--", label=f"{name} val")
        ax.set_title(term); ax.set_xlabel("epoch")
    axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
experiment("reference")

In [ ]:
experiment("3 epochs")          # then "from scratch", "lr too high", ... one run per cell

In [ ]:
plot_losses("reference", "3 epochs")

**Diagnose each run.** Copy the table into your notes and fill it in:

| recipe | your prediction | mAP50-95 | what the loss curves show | diagnosis |
|---|---|---|---|---|
| 3 epochs | | | | |
| from scratch | | | | |
| lr too high | | | | |
| lr too low | | | | |
| tiny images | | | | |
| no augmentation | | | | |
| head only | | | | |

- **3 epochs / lr too low**: both score low. How do the curves tell "stopped too early"
  apart from "barely moving"?
- **from scratch**: the same architecture and the same 381 images. What did 118k COCO
  images bring to the reference?
- **lr too high**: what do the losses do? Is `best.pt` saved from a good epoch?
- **tiny images**: at 160 px the stride-32 grid is 5 × 5. Which classes lose the most AP?
- **no augmentation**: compare the gap between train and val `box_loss` with the
  reference. Is 10 epochs long enough to see overfitting? Rerun at 30 to check.
- **head only**: far fewer weights learn. Faster, worse, or both? When would you freeze?

## 10. Inside the detector: the raw output

`predict` hides one tensor. Call the network yourself on the most crowded validation image
and decode it by hand, as in the lesson: a score per location and class, a threshold, then NMS.

In [ ]:
import cv2, torchvision
from ultralytics.data.augment import LetterBox

net = model.model.eval()
iid = max(split["val"], key=lambda i: len(val_gt[i]))            # the most crowded validation image
img = cv2.imread(f"yolo/val/images/{iid}.jpg")
x = LetterBox((640, 640), auto=False)(image=img)                  # resize and pad to 640 x 640, as predict does
x = torch.from_numpy(x[:, :, ::-1].copy()).permute(2, 0, 1)[None].float() / 255
with torch.no_grad():
    y = net(x.to(next(net.parameters()).device))[0].cpu()
print("raw output", tuple(y.shape))                               # (1, 4 + 7, 8400)

boxes, class_scores = y[0, :4].T, y[0, 4:].T                      # 8400 x 4 (cx, cy, w, h in pixels), 8400 x 7
best_score, best_class = class_scores.max(1)
levels = {8: (0, 6400), 16: (6400, 8000), 32: (8000, 8400)}       # stride: slice of the 8400 locations
for stride, (a, b) in levels.items():
    print(f"stride {stride:2d}: {b - a:5d} locations, {(best_score[a:b] > 0.25).sum().item():3d} scoring above 0.25")

keep = best_score > 0.25
xyxy = torchvision.ops.box_convert(boxes[keep], "cxcywh", "xyxy")
kept = torchvision.ops.batched_nms(xyxy, best_score[keep], best_class[keep], iou_threshold=0.7)
n_predict = len(model.predict(f"yolo/val/images/{iid}.jpg", conf=0.25, iou=0.7, device=device, verbose=False)[0].boxes)
print(f"{len(val_gt[iid])} animals; hand-decoded {len(kept)} boxes; model.predict {n_predict}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(x[0].permute(1, 2, 0)); axes[0].set_title("network input"); axes[0].axis("off")
for ax, (stride, (a, b)) in zip(axes[1:], levels.items()):
    side = 640 // stride
    ax.imshow(best_score[a:b].reshape(side, side), vmin=0, vmax=1, cmap="magma")
    ax.set_title(f"stride {stride}: {side} x {side} grid, best class score"); ax.axis("off")
plt.tight_layout(); plt.show()

**Questions**
- Where does 8400 come from? Compare with YOLOv1's 98 boxes.
- Which stride fires on the small fish, which on a large shark or a penguin?
- YOLOv1 had an objectness confidence $\hat{C}$. Where is it in this tensor?
- Your count and `predict`'s can differ by a box or two. `predict` pads to the smallest
  rectangle, not to a square: why would that move a box across the NMS or the threshold?

## 11. A YOLOv1 you can read (about 5 minutes on a T4)

The lesson's YOLOv1, in about 80 lines: a pretrained ResNet-18 backbone, an
$S \times S \times (5B + C)$ head, the target assignment, the five-term loss and the
decoding, scored by the leaderboard's metric. It is not meant to beat YOLOv8n; it is meant
to be read, line by line, against the lesson. Two changes from the paper, both from v2:
a conv head instead of the two FC layers, and sigmoids that keep every box number in
$[0, 1]$.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

B, C, IMG = 2, len(class_names), 448
dev = "cuda" if torch.cuda.is_available() else "cpu"

class MiniYOLO(nn.Module):
    """ResNet-18 backbone (ImageNet) + a conv head that outputs S x S x (5B + C)."""
    def __init__(self, S):
        super().__init__()
        resnet = torchvision.models.resnet18(weights="IMAGENET1K_V1")
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])   # 512 x 14 x 14 for a 448 input
        self.head = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.1),
            nn.AdaptiveAvgPool2d(S),                                     # the S x S grid
            nn.Conv2d(256, 5 * B + C, 1))

    def forward(self, x):
        out = self.head(self.backbone(x)).permute(0, 2, 3, 1)           # N x S x S x (5B + C)
        boxes = out[..., :5 * B].sigmoid()                               # x, y, w, h, confidence
        classes = out[..., 5 * B:].softmax(-1)                           # Pr(class | object), one per cell
        return torch.cat([boxes, classes], -1)


def encode(gt, S):
    """(n, 5) labels -> S x S x (5 + C) target: x*, y*, w*, h*, object, one-hot class; and how many boxes it dropped."""
    t, dropped = torch.zeros(S, S, 5 + C), 0
    for c, xc, yc, w, h in gt:
        col, row = min(int(xc * S), S - 1), min(int(yc * S), S - 1)    # the responsible cell
        if t[row, col, 4] == 1:                                          # a second centre in the same cell
            dropped += 1
            continue
        t[row, col, :5] = torch.tensor([xc * S - col, yc * S - row, w, h, 1.0])
        t[row, col, 5 + int(c)] = 1
    return t, dropped


def iou_in_cell(a, b, S):
    """IoU of two boxes of the same cell: (x, y) in cell units, (w, h) in image units."""
    def corners(z):
        cx, cy = z[..., 0] / S, z[..., 1] / S
        return cx - z[..., 2] / 2, cy - z[..., 3] / 2, cx + z[..., 2] / 2, cy + z[..., 3] / 2
    ax1, ay1, ax2, ay2 = corners(a)
    bx1, by1, bx2, by2 = corners(b)
    inter = (torch.min(ax2, bx2) - torch.max(ax1, bx1)).clamp(0) * (torch.min(ay2, by2) - torch.max(ay1, by1)).clamp(0)
    return inter / ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter + 1e-9)


def yolo_loss(pred, target, S, l_coord=5.0, l_noobj=0.5):
    """The five terms of the YOLOv1 loss, summed over the batch and divided by its size."""
    N = pred.shape[0]
    boxes = pred[..., :5 * B].reshape(N, S, S, B, 5)
    obj = target[..., 4] == 1                                            # cells holding an object centre
    ious = torch.stack([iou_in_cell(boxes[..., j, :4], target[..., :4], S) for j in range(B)], -1)
    resp = F.one_hot(ious.argmax(-1), B).bool() & obj[..., None]        # the responsible predictor
    r, t = boxes[resp], target[obj][:, :4]
    terms = {
        "xy":    l_coord * ((r[:, :2] - t[:, :2]) ** 2).sum(),
        "wh":    l_coord * ((r[:, 2:4].sqrt() - t[:, 2:4].sqrt()) ** 2).sum(),
        "obj":   ((r[:, 4] - ious[resp].detach()) ** 2).sum(),           # confidence target = current IoU
        "noobj": l_noobj * (boxes[..., 4][~resp] ** 2).sum(),
        "cls":   ((pred[..., 5 * B:][obj] - target[..., 5:][obj]) ** 2).sum(),
    }
    return sum(terms.values()) / N, {k: v.item() / N for k, v in terms.items()}


def decode(out, S, conf=0.001, iou=0.5, max_det=300):
    """One image's S x S x (5B + C) output -> (k, 6) detections: class, score, xc, yc, w, h."""
    rows, cols = torch.meshgrid(torch.arange(S), torch.arange(S), indexing="ij")
    boxes = out[..., :5 * B].reshape(S, S, B, 5)
    xc = (cols[..., None] + boxes[..., 0]) / S                           # cell offset -> image coordinates
    yc = (rows[..., None] + boxes[..., 1]) / S
    xywh = torch.stack([xc, yc, boxes[..., 2], boxes[..., 3]], -1).reshape(-1, 4)
    scores = (boxes[..., 4:5] * out[..., None, 5 * B:]).reshape(-1, C)   # Pr(c | obj) x confidence
    i, c = (scores > conf).nonzero(as_tuple=True)
    s, xywh = scores[i, c], xywh[i]
    keep = torchvision.ops.batched_nms(torchvision.ops.box_convert(xywh, "cxcywh", "xyxy"), s, c, iou)[:max_det]
    return torch.column_stack([c[keep].float(), s[keep], xywh[keep]]).numpy()

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

def load(path):
    img = Image.open(path).convert("RGB").resize((IMG, IMG))           # labels are normalised: a resize keeps them valid
    return (torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255 - MEAN) / STD

X = torch.stack([load(f"data/train/images/{i}.jpg") for i in split["train"]])
Y = [read_labels(i) for i in split["train"]]
X_val = torch.stack([load(f"data/train/images/{i}.jpg") for i in split["val"]])

for S in (7, 14):
    dropped = sum(encode(y, S)[1] for y in Y)
    total = sum(len(y) for y in Y)
    print(f"S = {S:2d}: {dropped} of {total} training boxes ({dropped / total:.0%}) share a cell and are never learnt")

In [ ]:
def train_mini(S=7, epochs=30, **loss_weights):
    torch.manual_seed(0)
    net = MiniYOLO(S).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=5e-4)
    for epoch in range(epochs):
        net.train()
        perm, terms = torch.randperm(len(X)), {}
        for k in range(0, len(X), 16):
            idx = perm[k:k + 16]
            flip = torch.rand(len(idx)) < 0.5                            # horizontal flip: x -> 1 - x
            xb = torch.where(flip[:, None, None, None], X[idx].flip(-1), X[idx])
            labels = [Y[i].copy() for i in idx.tolist()]
            for lab, f in zip(labels, flip.tolist()):
                if f:
                    lab[:, 1] = 1 - lab[:, 1]
            yb = torch.stack([encode(lab, S)[0] for lab in labels])
            loss, parts = yolo_loss(net(xb.to(dev)), yb.to(dev), S, **loss_weights)
            opt.zero_grad(); loss.backward(); opt.step()
            for name, v in parts.items():
                terms[name] = terms.get(name, 0) + v * len(idx) / len(X)
        if (epoch + 1) % 5 == 0:
            print(f"epoch {epoch + 1:2d}  " + "  ".join(f"{k} {v:6.2f}" for k, v in terms.items()))
    net.eval()
    with torch.no_grad():
        out = torch.cat([net(X_val[k:k + 16].to(dev)).cpu() for k in range(0, len(X_val), 16)])
    pred = {iid: decode(o, S) for iid, o in zip(split["val"], out)}
    s = dm.evaluate(val_gt, pred)
    print(f"S = {S}: validation mAP50-95 {s['map50_95']:.4f}   mAP50 {s['map50']:.4f}")
    return net, pred

mini, mini_pred = train_mini(S=7)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, iid in enumerate(split["val"][:3]):
    draw(axes[0, col], f"yolo/val/images/{iid}.jpg", val_pred[iid][val_pred[iid][:, 1] >= 0.25], "YOLOv8n, score >= 0.25")
    draw(axes[1, col], f"yolo/val/images/{iid}.jpg", mini_pred[iid][mini_pred[iid][:, 1] >= 0.25], "mini YOLOv1, score >= 0.25")
plt.tight_layout(); plt.show()

**Questions**
- At $S = 7$, almost a quarter of the training boxes are never learnt. Which row of the
  lesson's *Limits of YOLOv1* table is that? Train `train_mini(S=14)`: what changes, in the
  dropped boxes and in the score?
- The `noobj` term starts as the largest. Why? Train with `l_noobj=5.0`, then `l_noobj=0.0`,
  and look at the confidences of the predicted boxes.
- Remove the square roots in the `wh` term. Which boxes get worse, small or large?
- The mini model reaches a fraction of YOLOv8n's mAP50-95. Rank the later changes of the
  lesson (anchors, three strides, anchor-free head, CIoU/DFL, mosaic) by how much of the gap
  you think each one closes, then test one of them.

## 12. Where to go from here

Change one thing, compare the validation mAP50-95, submit only if it improved.

- **Train longer, or bigger.** `epochs=60`; `yolov8s.pt` or `yolo11s.pt` instead of the nano.
  Bigger models are more accurate and slower: is the gain worth it?
- **Augmentation.** Ultralytics already applies mosaic, flips and colour jitter: look at
  `train_batch0.jpg` in the run folder, then try `degrees`, `scale` or `mixup`.
- **Rare classes.** mAP averages over the 7 classes: the few puffins and stingrays weigh as
  much as the hundreds of fish. Which class AP is lowest, and why?
- **Use all the data.** Once the settings are chosen, retrain on all 448 images before the
  final submission.
- **Look at the errors.** Draw the validation images with the lowest recall: missed small
  objects, duplicate boxes, one class taken for another?